# Partie 1 - Préparation des données

### Importation des librairies
NumPy permet la manipulation de tableaux et de matrices.

In [2]:
import numpy as nmp

### Importation de Pandas
Pandas permet de manipuler des données sous forme de tableaux (DataFrame). On configure aussi l'affichage pour voir toutes les colonnes.

In [3]:
import pandas as pd

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('future.no_silent_downcasting', True)

/tmp/ipykernel_28910/1190158754.py:4: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


### Chargement du Pokédex
On charge le fichier `pokedex.csv` dans un DataFrame nommé `nosPokemons`.

In [5]:
nosPokemons = pd.read_csv("datas/pokedex.csv")

#Affichage des colonnes du Dataframe
print(nosPokemons.columns.values)

<StringArray>
[                 'NUMERO',                     'NOM',
                  'TYPE_1',                  'TYPE_2',
           'POINTS_DE_VIE',          'NIVEAU_ATTAQUE',
          'NIVEAU_DEFENSE', 'NIVEAU_ATTAQUE_SPECIALE',
 'NIVEAU_DEFENSE_SPECIALE',                 'VITESSE',
      'NOMBRE_GENERATIONS',              'LEGENDAIRE']
Length: 12, dtype: str


# Affichage des dix premières lignes de nos données

Pour afficher les dix premières lignes de nos données, nous allons utiliser la fonction Head du Dataframe nosPokemons que nous avons créé.

In [6]:
print(nosPokemons.head(10))  # Affiche les 10 premières lignes du DataFrame

   NUMERO               NOM TYPE_1  TYPE_2  POINTS_DE_VIE  NIVEAU_ATTAQUE  \
0       1        Bulbizarre  Herbe  Poison             45              49   
1       2        Herbizarre  Herbe  Poison             60              62   
2       3        Florizarre  Herbe  Poison             80              82   
3       4   Mega Florizarre  Herbe  Poison             80             100   
4       5         Salameche    Feu     NaN             39              52   
5       6         Reptincel    Feu     NaN             58              64   
6       7         Dracaufeu    Feu     Vol             78              84   
7       8  Mega Dracaufeu X    Feu  Dragon             78             130   
8       9  Mega Dracaufeu Y    Feu     Vol             78             104   
9      10          Carapuce    Eau     NaN             44              48   

   NIVEAU_DEFENSE  NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE  VITESSE  \
0              49                       65                       65     

### Transformation de la feature LEGENDAIRE
La feature `LEGENDAIRE` indique si un Pokémon est légendaire (VRAI) ou non (FAUX). Être légendaire est rare et donne plus de force, donc c'est une information utile.

Par contre, dans son état actuel, elle est lue comme une **chaîne de caractères**. Le Machine Learning ne fonctionne qu'avec des données **numériques**, donc on la transforme :
- `VRAI` → `1`
- `FAUX` → `0`

In [7]:
# remplacement de VRAI FAUX par 1 0
nosPokemonsClean = nosPokemons
nosPokemonsClean["LEGENDAIRE"] = nosPokemonsClean["LEGENDAIRE"].replace("VRAI", 1)
nosPokemonsClean["LEGENDAIRE"] = nosPokemonsClean["LEGENDAIRE"].replace("FAUX", 0)
nosPokemonsClean["LEGENDAIRE"] = nosPokemonsClean["LEGENDAIRE"].astype(int)
print(nosPokemonsClean.head())

   NUMERO              NOM TYPE_1  TYPE_2  POINTS_DE_VIE  NIVEAU_ATTAQUE  \
0       1       Bulbizarre  Herbe  Poison             45              49   
1       2       Herbizarre  Herbe  Poison             60              62   
2       3       Florizarre  Herbe  Poison             80              82   
3       4  Mega Florizarre  Herbe  Poison             80             100   
4       5        Salameche    Feu     NaN             39              52   

   NIVEAU_DEFENSE  NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE  VITESSE  \
0              49                       65                       65       45   
1              63                       80                       80       60   
2              83                      100                      100       80   
3             123                      122                      120       80   
4              43                       60                       50       65   

   NOMBRE_GENERATIONS  LEGENDAIRE  
0                   1     

### Manque-t-il des données ?
Maintenant que la feature `LEGENDAIRE` est transformée, on vérifie combien d'observations contient le jeu de données.

La propriété `shape` du DataFrame donne le nombre de **lignes** (observations, donc les Pokémon) et de **colonnes** (features).

In [8]:
#Comptage du nombre d'observations et de features
print(nosPokemonsClean.shape)

(800, 12)


Le jeu de données contient **800 observations** (Pokémon) et **12 features**, ce qui correspond aux 12 colonnes affichées plus haut.

### Y a-t-il des observations manquantes ?
La fonction `info()` donne, pour chaque feature, le nombre de valeurs **non nulles** ainsi que le **type** des données. Ça permet de repérer les valeurs manquantes.

In [9]:
#Informations sur notre jeu de données
print(nosPokemonsClean.info())

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   NUMERO                   800 non-null    int64
 1   NOM                      799 non-null    str  
 2   TYPE_1                   800 non-null    str  
 3   TYPE_2                   413 non-null    str  
 4   POINTS_DE_VIE            800 non-null    int64
 5   NIVEAU_ATTAQUE           800 non-null    int64
 6   NIVEAU_DEFENSE           800 non-null    int64
 7   NIVEAU_ATTAQUE_SPECIALE  800 non-null    int64
 8   NIVEAU_DEFENSE_SPECIALE  800 non-null    int64
 9   VITESSE                  800 non-null    int64
 10  NOMBRE_GENERATIONS       800 non-null    int64
 11  LEGENDAIRE               800 non-null    int64
dtypes: int64(9), str(3)
memory usage: 75.1 KB
None


On constate que :
- le jeu de données contient **800 observations** et **12 features**;
- il manque **1 nom** (799 valeurs sur 800);
- il manque **387 `TYPE_2`** (413 valeurs sur 800). C'est normal, car beaucoup de Pokémon n'ont qu'un seul type.

On voit aussi le type de chaque feature :
- `int64` : nombres entiers (ex. `POINTS_DE_VIE`, `VITESSE`);
- `object` : chaînes de caractères (`NOM`, `TYPE_1`, `TYPE_2`).

### À la recherche des features manquantes
Pour `TYPE_2`, les valeurs manquantes sont **normales** : un Pokémon peut ne pas avoir de type secondaire. On les laisse comme elles sont.

Pour le **nom manquant**, on ne sait pas encore si le nom d'un Pokémon a une influence sur l'issue des combats. Comme on ne peut pas l'exclure, on ne laisse pas cette valeur vide.

On commence par trouver l'observation dont la feature `NOM` est vide (`isnull()`).

In [10]:
print(nosPokemonsClean[nosPokemonsClean['NOM'].isnull()])

    NUMERO  NOM  TYPE_1 TYPE_2  POINTS_DE_VIE  NIVEAU_ATTAQUE  NIVEAU_DEFENSE  \
62      63  NaN  Combat    NaN             65             105              60   

    NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE  VITESSE  \
62                       60                       70       95   

    NOMBRE_GENERATIONS  LEGENDAIRE  
62                   1           0  


Le nom manquant se trouve à l'**observation 62** (Pokémon numéro 63, de type Combat).

Il existe deux façons de traiter une observation manquante :
- **supprimer** l'observation;
- **remplacer** la valeur par une valeur calculée ou par la vraie valeur.

Comme tous les Pokémon sont référencés dans le [Pokédex national](https://www.pokepedia.fr/Liste_des_Pok%C3%A9mon_dans_l%27ordre_du_Pok%C3%A9dex_National), on peut retrouver le vrai nom et le remplacer.